In [1]:
import syft as sy
print(sy.__version__)

0.9.5


In [2]:
data_site = sy.orchestra.launch(name="Glaucoma-research-centre")

CRITICAL:syft.server.server:Hash of the signing key '5e298...'


Using SQLiteDBConfig and sqlite:////tmp/syft/c0ac1c82615848859d8d7ae2e5ee1548/db/c0ac1c82615848859d8d7ae2e5ee1548_json.db


SyftInfo: You have launched a development server at http://0.0.0.0:None. It is intended only for local use.

In [3]:
client = data_site.login(email="rachel@datascience.inst", password="syftrocks")

Logged into <Glaucoma-research-centre: High side Datasite> as <rachel@datascience.inst>


In [4]:
# 2. Retrieve the dataset and the asset pointer
dataset = client.datasets["Medical Images Dataset"]
asset = dataset.assets["classification_images"]

In [5]:
# This is the pointer to the raw data path on the server
data_pointer = asset.pointer
print("Pointer to raw data on the server:", data_pointer)

Pointer to raw data on the server: {'images': array([[[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        ...,

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],

        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]]],


       [[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
        

In [6]:
# 3. Create a new PySyft Project
# Now we explicitly include the 'members' argument
my_project = sy.Project(
    name="My TFG Project",
    description="Training a classification model using remote data preprocessing.",
    members=[client]
)


In [7]:
@sy.syft_function(input_policy=sy.ExactMatch(raw_data_dict=asset)) 
def remote_preprocessing_and_training(raw_data_dict):
    import traceback 
    
    try:
        from data_preprocessing import Data_Preprocessing

        # 1. Instanciamos tu preprocesador con la ruta real
        actual_path = raw_data_dict["path"]
        preprocessor = Data_Preprocessing(
            data_path=actual_path,
            prep_batch_size=32 
        )
        
        # 2. Accedemos al dataset de Hugging Face ya construido
        ready_dataset = preprocessor.dataset
        
        # 3. Extraemos la información que pidió el tutor
        num_imagenes = len(ready_dataset)
        
        # 4. Devolvemos el diccionario seguro con TODA la info
        return {
            "status": "Exito",
            "total_imagenes": num_imagenes,
            "error": None
        }
        
    except Exception as e:
        return {"status": "CRASH INTERNO", "error": traceback.format_exc()}

SyftSuccess: Syft function 'remote_preprocessing_and_training' successfully created. To add a code request, please create a project using `project = syft.Project(...)`, then use command `project.create_code_request`.

In [8]:
# @sy.syft_function(input_policy=sy.ExactMatch(raw_data_dict=asset)) 
# def remote_preprocessing_and_training(raw_data_dict):
#     import traceback # Importamos esto DENTRO de la función remota
    
#     try:
#         from data_preprocessing import Data_Preprocessing

#         actual_path = raw_data_dict["path"]
        
#         preprocessor = Data_Preprocessing(
#             data_path=actual_path,
#             prep_batch_size=32 
#         )

#         print("Starting remote preprocessing and trainingv4...")
        
#         # Eliminamos la referencia compleja por si acaso
#         # del preprocessor
        
#         return {"status": "Completado con éxito", "error": None}
        
#     except Exception as e:
#         # ¡AQUÍ ESTÁ LA TRAMPA! 
#         # Si explota, capturamos el error de Python y lo devolvemos como un string.
#         # PySyft pensará que la función ha terminado bien y nos dejará ver esto.
#         error_str = traceback.format_exc()
#         return {"status": "CRASH INTERNO", "error": error_str}

In [9]:
# 5. Submit the project and the code request to the server
# The API syntax for submitting projects can vary slightly depending on your 
# PySyft 0.8.x version, but the standard flow requires adding the request to the project.
my_project.create_code_request(obj=remote_preprocessing_and_training, client=client)

print("Project created and code execution requested. Waiting for Data Owner approval...")

Project created and code execution requested. Waiting for Data Owner approval...


In [13]:
client = data_site.login(email="rachel@datascience.inst", password="syftrocks")

Logged into <Glaucoma-research-centre: High side Datasite> as <rachel@datascience.inst>


In [14]:
# 4. Access the approved code from your project and execute it!
# Notice we pass the pointer as the argument, just like we defined in the function
# CORRECT
approved_function = client.code.remote_preprocessing_and_training


In [15]:
result_pointer = approved_function(raw_data_dict=asset)

print("Code is executing on the remote server...")

Resolving data files:   0%|          | 0/81 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Map (num_proc=1):   0%|          | 0/80 [00:00<?, ? examples/s]

Code is executing on the remote server...


In [16]:
# 5. Download the final result to your local machine
# The .get() method is what actually transfers the allowed output 
# (the dictionary with the count) from the server to your laptop.
final_result = result_pointer.get()

print("Execution finished! Here are the results:")
print(final_result)

Execution finished! Here are the results:
{'status': 'Exito', 'total_imagenes': 80, 'error': None}


In [ ]:
data_site.land()